In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# Automatically use Kaggle's GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# REPLACE THIS PATH with the one you copied from the Kaggle sidebar
file_path = '/kaggle/input/datasets/soumilnegi154/shakespeare/shakespeare.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
char2int = {ch: i for i, ch in enumerate(chars)}
int2char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)

print(f"Total characters: {len(text)}, Unique characters: {vocab_size}")

encoded_text = [char2int[ch] for ch in text]
seq_length = 100 
X_data = []
y_data = []

for i in range(0, len(encoded_text) - seq_length):
    X_data.append(encoded_text[i : i + seq_length])
    y_data.append(encoded_text[i + 1 : i + seq_length + 1]) 

X = torch.tensor(X_data, dtype=torch.long)
y = torch.tensor(y_data, dtype=torch.long)

batch_size = 128
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

class ShakespeareLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super(ShakespeareLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, hidden):
        embedded = self.embedding(x) 
        out, hidden = self.lstm(embedded, hidden)
        out = out.reshape(-1, self.hidden_size)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        hidden = (torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device),
                  torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device))
        return hidden

embedding_dim = 64
hidden_size = 256
num_layers = 2
epochs = 10  # You can increase this since Kaggle GPUs are fast
learning_rate = 0.001

model = ShakespeareLSTM(vocab_size, embedding_dim, hidden_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print("Starting Training...")
for epoch in range(epochs):
    model.train()
    
    for batch_idx, (inputs, targets) in enumerate(dataloader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        current_batch_size = inputs.size(0)
        hidden = model.init_hidden(current_batch_size)
        
        optimizer.zero_grad()
        outputs, hidden = model(inputs, hidden)
        loss = criterion(outputs, targets.view(-1))
        
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        optimizer.step()
        
        if batch_idx % 200 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(dataloader)}] | Loss: {loss.item():.4f}")

def generate_text(model, start_str, length, temperature=0.8):
    model.eval()
    hidden = model.init_hidden(1)
    
    input_seq = torch.tensor([char2int[ch] for ch in start_str], dtype=torch.long).unsqueeze(0).to(device)
    
    for i in range(len(start_str) - 1):
        _, hidden = model(input_seq[:, i].unsqueeze(1), hidden)
        
    current_char = input_seq[:, -1].unsqueeze(1)
    generated_text = start_str
    
    with torch.no_grad():
        for _ in range(length):
            output, hidden = model(current_char, hidden)
            output = output / temperature
            probs = torch.softmax(output, dim=1).squeeze()
            char_idx = torch.multinomial(probs, 1).item()
            generated_text += int2char[char_idx]
            current_char = torch.tensor([[char_idx]], dtype=torch.long).to(device)
            
    return generated_text

print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=0.8))

Using device: cuda
Total characters: 1115394, Unique characters: 65
Starting Training...
Epoch [1/10] | Batch [0/8714] | Loss: 4.1945
Epoch [1/10] | Batch [200/8714] | Loss: 2.1284
Epoch [1/10] | Batch [400/8714] | Loss: 1.8453
Epoch [1/10] | Batch [600/8714] | Loss: 1.6898
Epoch [1/10] | Batch [800/8714] | Loss: 1.6463
Epoch [1/10] | Batch [1000/8714] | Loss: 1.5463
Epoch [1/10] | Batch [1200/8714] | Loss: 1.4807
Epoch [1/10] | Batch [1400/8714] | Loss: 1.4730
Epoch [1/10] | Batch [1600/8714] | Loss: 1.4485
Epoch [1/10] | Batch [1800/8714] | Loss: 1.4115
Epoch [1/10] | Batch [2000/8714] | Loss: 1.4088
Epoch [1/10] | Batch [2200/8714] | Loss: 1.3758
Epoch [1/10] | Batch [2400/8714] | Loss: 1.3561
Epoch [1/10] | Batch [2600/8714] | Loss: 1.3516
Epoch [1/10] | Batch [2800/8714] | Loss: 1.3313
Epoch [1/10] | Batch [3000/8714] | Loss: 1.3141
Epoch [1/10] | Batch [3200/8714] | Loss: 1.3057
Epoch [1/10] | Batch [3400/8714] | Loss: 1.3091
Epoch [1/10] | Batch [3600/8714] | Loss: 1.2768
Epoch 

In [2]:
print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=0.7))

O Romeo, Romeo! wherefore art thou deserves
That now the best traitor to the senate doth?

Third Musician:
Then have you never spoke of it? Shame our kind of love.

ROMEO:
Not light, and give me this I thank his husband.

KING RICHARD II:
What make you tongue doth in this another
To be as much of joy?

DERBY:
He should have had me die that look'd and heaven
With diseases as well as intent in the
traitor, never bears upon him, where he shall be
The wrong with Calibans: so if come son--
For-word is one of those Duke of York, the pe


In [3]:
print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=0.1))

O Romeo, Romeo! wherefore art thou not a man?

BUCKINGHAM:
What, will you not speak? O my mother knows not so?

Second Keeper:
Here in this contract, I have not deserved
The prince my father with his complexion.

QUEEN:
'Tis nothing but some other sight, and there
I never seek to be so on out of death.

DUKE VINCENTIO:
The gods be good to me in his complices:
You have made good work, when he wakes with the world.

GLOUCESTER:
Then let them be sworn that world with all the state,
Which now a shopt of heaven with silence with the w


In [4]:
print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=0.5))

O Romeo, Romeo! wherefore art thou that?

BENVOLIO:
Here comes the house of Lancaster.

DUKE VINCENTIO:
You are pleasant, sir; for then you should not be
content: but if you do her take away on mine own.

First Senator:
Stay, you would not say 'thare to do it.

PROSPERO:
Before thou liest.

Second Murderer:
O my sight, sir, come upon these good success.
They follows me about the common prince,
And this far from hence straight die for them:
But come, sir, a word with you, and I will do it;
I do remember what they can follow thee.



In [5]:
print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=0.9))

O Romeo, Romeo! wherefore art thou such a garment?

RICHARD:
'Twas most five under your goods must I see,
Unto a place o' thing set on foot, stays that thought,
That ne'er supposed methink unto the beauty;
And when the very princely life thou mayst,
Once profence a child, thou wilt be a maid in
story to my turn; then is not death: 'tis a sword for busies!

Second Murderer:
No more! 'Tis this to that mathem! I never heard me hollow.

LUCENTIO:
Who shall none of this afternoon? I have done
untaked him: I'll understand your weeping 


In [6]:
print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=1))

O Romeo, Romeo! wherefore art thou that slaughter,
As one that born in being so and to fight?

KING HENRY VI:
Far never-hearted Hereford! Who knows not not
But as if it be, mistressman?

Lord:
Say, sir, you are.

CORIOLANUS:
It is Angelo, that fong is grown:
Nicely to resist her well.

MIRANDA:
Yield,
Like a best knaves, and not yet we will cut out.
Some part Geirsh like the language and thou art full of age.
For shumour will I say my son, Tybalt art,
Though my honour, you are virtue: look then barls,
To core on your uncle: there


In [11]:
print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=1))

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
